In [2]:
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
import seaborn as sns
from scipy.stats import ttest_ind

lat_col = "network_latency"
bw_col = "network_bandwidth"
time_col = "time"
alg_col = "algorithm"
len_col = "prompt_length"

def get_experiment_df(voltage, voltage_improv):
    df_voltage = pd.read_json("../results/" + voltage)
    df_voltage[alg_col] = "voltage"


    df_voltage_improv = pd.read_json("../results/" + voltage_improv)
    df_voltage_improv[alg_col] = "voltage_improv"

    df = pd.concat([df_voltage, df_voltage_improv], ignore_index=True)

    df[bw_col] = df[bw_col].apply(lambda x: x * 1e-6)  # convert to Mbps
    df[lat_col] = df[lat_col].apply(lambda x: x * 1e3)  # convert to ms

    # Clean data that has warmup effects
    # Find the mean std for each lat_col, bw_col and exclude all entries that are outside 2 stds
    def clean_warmup_effects(df):
        clean_dfs = []
        for (lat, bw, alg, length), group in df.groupby([lat_col, bw_col, alg_col, len_col]):
            time_mean = group[time_col].mean()
            time_std = group[time_col].std()
            # print(f"For lat: {lat}, bw: {bw}, alg: {alg}, length: {length} => mean: {time_mean}, std: {time_std}")
            clean_group = group[(group[time_col] >= time_mean - 8 * time_std) & (group[time_col] <= time_mean + 8 * time_std)]
            clean_dfs.append(clean_group)
        return pd.concat(clean_dfs, ignore_index=True)

    df = clean_warmup_effects(df)
    return df

In [3]:
def plot_experiment_df(df):
    # Use seaborn barplot so we get mean + confidence intervals (default ci=95)
    for text_len in df[len_col].unique():
        df_len = df[df[len_col] == text_len]

        # Helpers for ordering
        def _to_num(x):
            try:
                return float(str(x).strip().lower().replace("ms", "").replace("s", ""))
            except:
                return x

        lat_order = sorted(df_len[lat_col].unique(), key=_to_num)
        alg_order = list(df_len[alg_col].dropna().unique())
        bw_vals = sorted(df_len[bw_col].unique(), key=_to_num)

        # determine y-limit from raw data (safer for CI visualization)
        ymax = df_len[time_col].max() * 1.10

        fig, axes = plt.subplots(1, len(bw_vals), figsize=(3 * len(bw_vals), 4), sharey=False)
        if len(bw_vals) == 1:
            axes = [axes]

        for ax, bw in zip(axes, bw_vals):
            sub = df_len[df_len[bw_col] == bw]

            sns.barplot(
                data=sub,
                x=lat_col,
                y=time_col,
                hue=alg_col,
                order=lat_order,
                hue_order=alg_order,
                ax=ax,
                errorbar=("ci", 95),
                capsize=0.06,
                palette="tab10",
                err_kws={"color": "k", "linewidth": 1},
            )

            ax.set_title(f"{bw} Mbps")
            ax.set_xlabel(lat_col + " (ms)")
            # ax.set_ylim(0, ymax)
            ax.grid(axis="y", alpha=0.3)
            ax.tick_params(axis="x", rotation=0)

        axes[0].set_ylabel(f"Mean {time_col} ± 95% CI")
        fig.suptitle(f"Text Length: {text_len}", fontsize=12)
        # single legend for the figure
        handles, labels = axes[-1].get_legend_handles_labels()
        fig.legend(handles, labels, title=alg_col, loc="upper right", ncol=len(alg_order))
        # remove per-axis legends
        for a in axes:
            a.legend_.remove()
        plt.tight_layout(rect=[0, 0, 1, 0.92])
        # Save as pdf
        plt.savefig(f"../results/voltage_analysis_text_length_{text_len}.pdf")
        plt.show()

In [11]:
def show_speedup_analysis(df):
    results = []

    for text_len in df[len_col].unique():
        df_len = df[df[len_col] == text_len]
        alg_order = sorted(df_len[alg_col].unique())
        for bw in df_len[bw_col].unique():
            for lat in df_len[lat_col].unique():
                sub = df_len[(df_len[lat_col] == lat) & (df_len[bw_col] == bw)]
                if len(sub[alg_col].unique()) < 2:
                    continue
                vals0 = sub[sub[alg_col] == alg_order[0]][time_col]
                vals1 = sub[sub[alg_col] == alg_order[1]][time_col]
                if len(vals0) > 1 and len(vals1) > 1:
                    t_stat, p_val = ttest_ind(vals0, vals1, equal_var=False)
                else:
                    p_val = float('nan')
                mean0 = vals0.mean()
                mean1 = vals1.mean()
                if not pd.isna(mean0) and not pd.isna(mean1) and len(vals0) > 0 and len(vals1) > 0:
                    # vals0_sorted = sorted(vals0)
                    # vals1_sorted = sorted(vals1)
                    # if len(vals0_sorted) >= 3 and len(vals1_sorted) >= 3:
                    #     mean_smallest_vals0 = sum(vals0_sorted[:3]) / 3
                    #     mean_largest_vals1 = sum(vals1_sorted[-3:]) / 3
                    #     mean_largest_vals0 = sum(vals0_sorted[-3:]) / 3
                    #     mean_smallest_vals1 = sum(vals1_sorted[:3]) / 3
                    #     best_speedup = mean_smallest_vals0 / mean_largest_vals1 if mean_largest_vals1 != 0 else float('nan')
                    #     worst_speedup = mean_largest_vals0 / mean_smallest_vals1 if mean_smallest_vals1 != 0 else float('nan')
                    # else:
                    #     best_speedup = min(vals0) / max(vals1) if max(vals1) != 0 else float('nan')
                    #     worst_speedup = max(vals0) / min(vals1) if min(vals1) != 0 else float('nan')
                    # if not pd.isna(best_speedup) and not pd.isna(worst_speedup):
                        # speedup = f"{(best_speedup-1)*100:.2f}% - {(worst_speedup-1)*100:.2f}%"
                    # else:
                        # speedup = "nan"
                    speedup = mean0 / mean1
                    results.append({
                        len_col: int(text_len),
                        bw_col: int(bw),
                        lat_col: int(lat),
                        f"{alg_order[0]}_mean_time": f"{mean0:.3f}",
                        f"{alg_order[1]}_mean_time": f"{mean1:.3f}",
                        "speedup": speedup,
                        "p_value": f"{p_val:.3f}"
                    })

    speedup_df = pd.DataFrame(results)
    # Export as csv
    speedup_df.to_csv("../results/voltage_speedup_analysis.csv", index=False)
    display(speedup_df)

In [12]:

cpu_2_df = get_experiment_df("quest/voltage_cpu_2.json", "quest/voltage_improv_cpu_2.json")

orin_cuda_2 = get_experiment_df("orin_voltage_4.json", "orin_voltage_improv_4.json")
orin_cuda_4 = get_experiment_df("orin_cuda_voltage_4.json", "orin_cuda_voltage_improv_4.json")

orin_sim_4_df = get_experiment_df("voltage_orin_sim_4.json", "voltage_improv_orin_sim_4.json")
orin_4_cpu_df = get_experiment_df("orin_voltage_2_cpu.json", "orin_voltage_improv_2_cpu.json")

quest_4_cuda_small = get_experiment_df("quest/voltage_cuda_4_2.json", "quest/voltage_improv_cuda_4_2.json")
quest_4_cuda_large = get_experiment_df("quest/voltage_cuda_4.json", "quest/voltage_improv_cuda_4.json")

merge_quest_4 = pd.concat([quest_4_cuda_small, quest_4_cuda_large], ignore_index=True)

quest_4_cuda_sim = get_experiment_df("quest/voltage_cuda_4_sim.json", "quest/voltage_improv_cuda_4_sim.json")
quest_4_cuda_sim_2 = get_experiment_df("quest/voltage_cuda_4_sim_2.json", "quest/voltage_improv_cuda_4_sim_2.json")


show_speedup_analysis(cpu_2_df)



,prompt_length,network_bandwidth,network_latency,voltage_mean_time,voltage_improv_mean_time,speedup,p_value
0,269,10,1,1.925,1.851,1.039927,0.461
1,269,10,5,1.895,1.831,1.034789,0.000
2,269,10,20,2.146,2.083,1.030358,0.000
3,269,100,1,1.350,1.299,1.039128,0.000
4,269,100,5,1.418,1.364,1.039605,0.000
5,269,100,20,1.676,1.615,1.037947,0.000
6,269,1000,1,1.309,1.294,1.011547,0.009
7,269,1000,5,1.379,1.320,1.045426,0.000
8,269,1000,20,1.632,1.573,1.037408,0.000
9,490,10,1,3.014,2.919,1.032383,0.000


In [13]:
orin_cuda_100_1_voltage = get_experiment_df("orin_cuda_100_1_voltage.json", "orin_cuda_100_1_voltage_improv.json")
orin_cuda_100_10_voltage = get_experiment_df("orin_cuda_100_10_voltage.json", "orin_cuda_100_10_voltage_improv.json")
orin_cuda_1000_1_voltage = get_experiment_df("orin_cuda_1000_1_voltage.json", "orin_cuda_1000_1_voltage_improv.json")

show_speedup_analysis(orin_cuda_100_1_voltage)
show_speedup_analysis(orin_cuda_100_10_voltage)
show_speedup_analysis(orin_cuda_1000_1_voltage)

,prompt_length,network_bandwidth,network_latency,voltage_mean_time,voltage_improv_mean_time,speedup,p_value
0,125,0,1000,0.679,0.717,0.945718,0.705
1,173,0,1000,0.840,0.833,1.008943,0.008
2,269,0,1000,1.253,1.246,1.004855,0.117
3,490,0,1000,2.139,2.148,0.995660,0.451
4,1002,0,1000,3.873,3.890,0.995593,0.568


,prompt_length,network_bandwidth,network_latency,voltage_mean_time,voltage_improv_mean_time,speedup,p_value
0,125,0,10000,1.343,1.277,1.051887,0.591
1,173,0,10000,1.364,1.351,1.009928,0.000
2,269,0,10000,1.710,1.693,1.010284,0.009
3,490,0,10000,2.451,2.432,1.007407,0.001
4,1002,0,10000,4.048,4.015,1.008104,0.221


,prompt_length,network_bandwidth,network_latency,voltage_mean_time,voltage_improv_mean_time,speedup,p_value
0,125,0,1000,0.422,0.437,0.966759,0.950
1,173,0,1000,0.317,0.317,1.001096,0.782
2,269,0,1000,0.428,0.437,0.979973,0.003
3,490,0,1000,0.642,0.656,0.978609,0.000
4,1002,0,1000,1.064,1.090,0.976483,0.416


In [14]:
orin_cpu_100_10 = get_experiment_df("orin_cpu_voltage_100_10.json", "orin_cpu_voltage_improv_100_10.json")
orin_cpu_1000_1 = get_experiment_df("orin_cpu_voltage_1000_1.json", "orin_cpu_voltage_improv_1000_1.json")
show_speedup_analysis(orin_cpu_100_10)
show_speedup_analysis(orin_cpu_1000_1)

,prompt_length,network_bandwidth,network_latency,voltage_mean_time,voltage_improv_mean_time,speedup,p_value
0,8,0,1000,1.729,1.787,0.967544,0.000
1,256,0,1000,12.226,11.937,1.024226,0.000


,prompt_length,network_bandwidth,network_latency,voltage_mean_time,voltage_improv_mean_time,speedup,p_value
0,1,0,1000,1.003,1.023,0.980333,0.393
1,2,0,1000,0.945,1.042,0.906590,0.000
2,4,0,1000,0.993,1.095,0.906164,0.000
3,8,0,1000,1.124,1.294,0.869276,0.000
4,16,0,1000,1.235,1.293,0.955601,0.001
5,32,0,1000,1.470,1.563,0.940666,0.000
6,64,0,1000,2.011,2.146,0.937096,0.000


In [19]:
voltage_sim = get_experiment_df("b_voltage.json", "b_voltage_improv.json")
show_speedup_analysis(voltage_sim)

,prompt_length,network_bandwidth,network_latency,voltage_mean_time,voltage_improv_mean_time,speedup,p_value
0,256,10,1,1.281,0.939,1.363578,0.000
1,256,10,5,1.574,1.188,1.325290,0.000
2,256,10,20,2.454,2.048,1.198080,0.000
3,256,100,1,0.329,0.256,1.282451,0.000
4,256,100,5,0.564,0.436,1.293138,0.000
5,256,100,20,1.465,1.129,1.297335,0.000
6,256,1000,1,0.232,0.198,1.173943,0.008
7,256,1000,5,0.465,0.370,1.254972,0.000
8,256,1000,20,1.301,1.105,1.177307,0.000
9,490,10,1,2.257,1.705,1.323659,0.000
